# Video Data Preprocessing — Jupyter Implementation (Start from Step 2)

This notebook implements a practical, beginner–intermediate video preprocessing pipeline using **OpenCV + NumPy + Matplotlib**.

**Steps covered:**
- Step 2: Frame extraction & sampling  
- Step 3: Inspect video properties  
- Step 4: Frame-wise preprocessing (image-based)  
- Step 5: Temporal preprocessing (frame differencing, temporal smoothing, basic optical flow)

In [ ]:

import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

def show_bgr(img_bgr, title="Frame", figsize=(6,4)):
    """Display a BGR image using matplotlib."""
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=figsize)
    plt.imshow(img_rgb)
    plt.title(title)
    plt.axis("off")
    plt.show()

def show_gray(img_gray, title="Gray", figsize=(6,4)):
    """Display a grayscale image."""
    plt.figure(figsize=figsize)
    plt.imshow(img_gray, cmap="gray")
    plt.title(title)
    plt.axis("off")
    plt.show()

def ensure_dir(path):
    os.makedirs(path, exist_ok=True)

VIDEO_PATH = "data-course/data-prep/video-data/video/car-detection.mp4"   # <-- replace with your video file path
OUT_DIR = "data-course/data-prep/video-data/video/output_frames"
ensure_dir(OUT_DIR)

cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise FileNotFoundError(f"Cannot open video: {VIDEO_PATH}")

print("Video opened successfully.")



### Step-2 Sampling Strategy: Take Every N-th Frame

In [ ]:
sample_every_n = 10  # keep 1 frame every 10 frames
sampled_frames = []
sampled_indices = []

frame_idx = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    if frame_idx % sample_every_n == 0:
        sampled_frames.append(frame)
        sampled_indices.append(frame_idx)
    
    frame_idx += 1

cap.release()

print(f"Total sampled frames: {len(sampled_frames)}")
print(f"First 10 sampled indices: {sampled_indices[:10]}")


### Visual Check (Sampled Frames)

In [ ]:

for i in range(min(3, len(sampled_frames))):
    show_bgr(sampled_frames[i], title=f"Sampled frame (index={sampled_indices[i]})")

### Save Sampled Frames (Optional)

In [ ]:

frames_dir = os.path.join(OUT_DIR, "sampled_frames")
ensure_dir(frames_dir)

for frame, idx in zip(sampled_frames[:20], sampled_indices[:20]):  # save first 20 only
    cv2.imwrite(os.path.join(frames_dir, f"frame_{idx:06d}.jpg"), frame)

print(f"Saved up to 20 sampled frames into: {frames_dir}")


### Step-3 Inspect Video Properties

In [ ]:
cap = cv2.VideoCapture(VIDEO_PATH)

fps = cap.get(cv2.CAP_PROP_FPS)
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
duration_sec = frame_count / fps if fps > 0 else None

cap.release()

print("Video Properties")
print("----------------")
print(f"FPS: {fps}")
print(f"Frame count: {frame_count}")
print(f"Resolution: {width} x {height}")
print(f"Duration (sec): {duration_sec}")


### Step-4 Frame-wise Preprocessing (Image-Based)

We will preprocess each sampled frame independently:

- resize
- convert to grayscale
- normalize
- histogram + statistics

In [ ]:
def preprocess_frame(frame_bgr, target_size=(224, 224), to_gray=True, normalize=True):
    # Resize
    resized = cv2.resize(frame_bgr, target_size, interpolation=cv2.INTER_AREA)
    
    if to_gray:
        gray = cv2.cvtColor(resized, cv2.COLOR_BGR2GRAY)
        out = gray
    else:
        out = resized
    
    if normalize:
        out = out.astype(np.float32) / 255.0  # range [0,1]
    
    return out

def frame_stats_gray(gray_01):
    """gray_01 expected in [0,1]."""
    arr = (gray_01 * 255.0).astype(np.uint8)
    return {
        "mean": float(np.mean(arr)),
        "std": float(np.std(arr)),
        "min": int(np.min(arr)),
        "max": int(np.max(arr)),
        "median": float(np.median(arr)),
    }

processed = []
stats_list = []

for f in sampled_frames:
    g = preprocess_frame(f, target_size=(224, 224), to_gray=True, normalize=True)
    processed.append(g)
    stats_list.append(frame_stats_gray(g))

print(f"Processed frames: {len(processed)}")
print("Example stats of first frame:", stats_list[0])


### Visualize Preprocessing Result

In [ ]:
# Show one example
idx0 = 0
show_bgr(sampled_frames[idx0], title="Original sampled frame")

gray_01 = processed[idx0]
show_gray((gray_01 * 255).astype(np.uint8), title="Preprocessed (grayscale, resized)")


### Histogram (Single Frame)

In [ ]:
arr = (processed[0] * 255).astype(np.uint8)

plt.figure(figsize=(6,4))
plt.hist(arr.ravel(), bins=256, range=[0,256])
plt.title("Histogram (Preprocessed Frame 0)")
plt.xlabel("Intensity")
plt.ylabel("Pixel Count")
plt.show()

### Quick Consistency Check Across Frames

We can monitor mean brightness over time (sampled indices).

In [ ]:
means = [s["mean"] for s in stats_list]

plt.figure(figsize=(8,4))
plt.plot(sampled_indices, means)
plt.title("Mean Intensity over Sampled Frames")
plt.xlabel("Original Frame Index")
plt.ylabel("Mean (0-255)")
plt.show()


### Step-5 Step 5 — Temporal Preprocessing (Time-Based)

Now we incorporate time by analyzing relationships between consecutive frames.

- Frame Differencing (Motion Cues)
- Temporal Smoothing
- Basic Optical Flow Visualization

### Frame Differencing (Motion Cues)


We compute differences on preprocessed grayscale frames.

$$D_t = |I_t - I_{t-1}|$$

In [ ]:
diffs = []
for i in range(1, len(processed)):
    d = cv2.absdiff((processed[i] * 255).astype(np.uint8),
                    (processed[i-1] * 255).astype(np.uint8))
    diffs.append(d)

print(f"Computed frame differences: {len(diffs)}")

i = 0
show_gray(diffs[i], title=f"Frame Difference: idx {sampled_indices[i+1]} - {sampled_indices[i]}")


### Temporal Smoothing (Reduce Flicker)

In [ ]:
def temporal_moving_average(frames_gray_01, window=5):
    """
    frames_gray_01: list of [0,1] grayscale frames (H,W)
    Returns smoothed list (same length; edges are handled by smaller windows).
    """
    smoothed = []
    for i in range(len(frames_gray_01)):
        start = max(0, i - window//2)
        end = min(len(frames_gray_01), i + window//2 + 1)
        stack = np.stack(frames_gray_01[start:end], axis=0)
        smoothed.append(np.mean(stack, axis=0))
    return smoothed

smoothed = temporal_moving_average(processed, window=5)

# Compare original vs smoothed for one frame
k = 10 if len(processed) > 10 else 0
show_gray((processed[k]*255).astype(np.uint8), title="Original preprocessed frame")
show_gray((smoothed[k]*255).astype(np.uint8), title="Temporally smoothed frame (moving average)")


### (Optional) Optical Flow (Basic)

In [ ]:
# Use two consecutive grayscale frames (uint8)
if len(processed) >= 2:
    prev = (processed[0] * 255).astype(np.uint8)
    nxt  = (processed[1] * 255).astype(np.uint8)

    flow = cv2.calcOpticalFlowFarneback(
        prev, nxt, None,
        pyr_scale=0.5, levels=3, winsize=15,
        iterations=3, poly_n=5, poly_sigma=1.2, flags=0
    )

    # Convert flow to magnitude for visualization
    mag, ang = cv2.cartToPolar(flow[...,0], flow[...,1])
    mag_norm = cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

    show_gray(mag_norm, title="Optical Flow Magnitude (Farneback)")
else:
    print("Not enough frames for optical flow.")


Learning Check (Optional)

1. Why might frame differencing fail when lighting changes suddenly?

2. Why does temporal smoothing help stability but risk blurring fast motion?